# 🧪 Laboratorio Didáctico: Algoritmos Find-S y Candidate-Elimination

Este notebook complementa la **Clase 2 de Teórico**.
Aquí exploraremos de forma interactiva y visual:
1. **Representación de hipótesis** y relaciones de orden general-específico ($\ge_g$).
2. **Algoritmo Find-S**: Búsqueda guiada por ejemplos positivos hacia la hipótesis más específica.
3. **Algoritmo Candidate-Elimination**: Mantenimiento del **Espacio de Versiones** ($VS_{H,D}$) a través de los límites $S$ (Específico) y $G$ (General).
4. **Clasificación de nuevas instancias** y resolución de ambigüedades.

In [ ]:
import sys
import os
# Asegurar que podamos importar desde el paquete algoritmos
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "..")))

import pandas as pd
import matplotlib.pyplot as plt
from algoritmos import Hypothesis, FindS, CandidateElimination, plot_version_space


--- 
## 1. Definición del Problema y Datos de Entrenamiento: ¿Cuándo salva Pedro un examen?

Definimos el dominio de los atributos discretos y cargamos el conjunto de entrenamiento $D$ presentado en el teórico.

In [ ]:
domains = {
    'Dedicación': ['Alta', 'Media', 'Baja'],
    'Dificultad': ['Alta', 'Media', 'Baja'],
    'Horario': ['Matutino', 'Nocturno'],
    'Humedad': ['Alta', 'Media', 'Baja'],
    'HumorDoc': ['Bueno', 'Malo']
}

X = [
    ['Alta', 'Alta', 'Nocturno', 'Media', 'Bueno'],
    ['Baja', 'Media', 'Matutino', 'Alta', 'Malo'],
    ['Media', 'Alta', 'Nocturno', 'Media', 'Malo'],
    ['Media', 'Alta', 'Matutino', 'Alta', 'Bueno']
]
y = ['SÍ', 'NO', 'SÍ', 'NO']

df_train = pd.DataFrame(X, columns=list(domains.keys()))
df_train['Pedro Salva? c(x)'] = y
df_train

--- 
## 2. Algoritmo Find-S

Find-S inicia en $h_0 = \langle \emptyset, \emptyset, \emptyset, \emptyset, \emptyset \rangle$ y se generaliza con cada ejemplo positivo ($c(x) = \text{SÍ}$), ignorando los negativos.

In [ ]:
find_s = FindS(num_attributes=len(domains), attribute_names=list(domains.keys()))
find_s.fit(X, y)

# Visualizar la traza paso a paso
history_fs = []
for rec in find_s.history:
    history_fs.append({
        'Paso': rec['step'],
        'Ejemplo procesado': str(rec['instance']) if rec['instance'] else 'Inicial',
        'Etiqueta': rec['label'] if rec['label'] else '-',
        'Acción': rec['action'],
        'Hipótesis h': str(rec['hypothesis'])
    })

pd.DataFrame(history_fs)

--- 
## 3. Algoritmo Candidate-Elimination y Espacio de Versiones

Candidate-Elimination mantiene los dos conjuntos frontera:
- $S$: Las hipótesis más específicas consistentes con los datos.
- $G$: Las hipótesis más generales consistentes con los datos.

In [ ]:
ce = CandidateElimination(domains)
ce.fit(X, y)

# Visualizar la evolución de S y G
history_ce = []
for rec in ce.history:
    history_ce.append({
        'Paso': rec['step'],
        'Tipo Ejemplo': rec['label'] if rec['label'] else 'Inicial',
        'Descripción': rec['description'],
        'Límite Específico (S)': str(rec['S']),
        'Límite General (G)': str(rec['G'])
    })

pd.DataFrame(history_ce)

### Visualización Gráfica del Espacio de Versiones Final

In [ ]:
plot_version_space(ce, title="Espacio de Versiones ($VS_{H,D}$) - Caso Pedro")

--- 
## 4. Clasificación de Nuevas Instancias (Diapositiva 25)

Probamos los 4 casos del examen presentados en el teórico para entender cómo clasifica el Espacio de Versiones:

In [ ]:
test_cases = [
    ["Alta", "Alta", "Nocturno", "Media", "Malo"],
    ["Alta", "Baja", "Matutino", "Alta", "Bueno"],
    ["Alta", "Alta", "Nocturno", "Baja", "Bueno"],
    ["Alta", "Baja", "Nocturno", "Media", "Bueno"]
]

results = []
for inst in test_cases:
    res = ce.classify(inst)
    results.append({
        'Dedicación': inst[0],
        'Dificultad': inst[1],
        'Horario': inst[2],
        'Humedad': inst[3],
        'HumorDoc': inst[4],
        'Veredicto': res['decision'],
        'Detalle': res['confidence']
    })

pd.DataFrame(results)

--- 
## 5. ¡Pruébalo con tus propias instancias!
Puedes modificar los valores de la siguiente celda para consultar cualquier caso nuevo:

In [ ]:
# Modifica estos valores para experimentar:
mi_instancia = ['Media', 'Alta', 'Nocturno', 'Media', 'Bueno']

pred = ce.classify(mi_instancia)
print(f"Instancia: {mi_instancia}")
print(f"Decisión:  {pred['decision']}")
print(f"Detalle:   {pred['confidence']}")